# ⚡ Notebook 2: Database Optimization for Writes

Before adding complexity, exhaust what your existing database can do. Proper tuning can often 10x your write throughput.

## Learning Objectives

By the end of this notebook, you'll understand:
- Write-optimized database patterns
- Index overhead and management
- Bulk insert techniques
- When to choose specialized databases

---

🔍 **Open Adminer** at http://localhost:8080 to watch write performance!

In [ ]:
import psycopg2
import psycopg2.extras
import time
import statistics
from concurrent.futures import ThreadPoolExecutor

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "writes_demo",
    "user": "demo",
    "password": "demo"
}

def get_connection():
    return psycopg2.connect(**DB_CONFIG)

print("✅ Connected to PostgreSQL")

## 📚 Write-Optimized Database Patterns

In [ ]:
print("📚 Database Types by Write Pattern")
print("=" * 60)
print("""
TRADITIONAL RDBMS (PostgreSQL, MySQL)
─────────────────────────────────────────────────────────────
• Updates data IN PLACE (requires disk seeks)
• Maintains B-tree indexes (rebalancing overhead)
• Strong consistency, ACID transactions
• Good for: Mixed workloads, complex queries
• Writes: ~1,000-50,000/sec depending on tuning

LOG-STRUCTURED (Cassandra, RocksDB)
─────────────────────────────────────────────────────────────
• APPEND-ONLY writes (sequential, fast!)
• Compaction merges data in background
• Eventually consistent (tunable)
• Good for: Write-heavy, simple queries
• Writes: ~10,000-100,000/sec

TIME-SERIES (InfluxDB, TimescaleDB)
─────────────────────────────────────────────────────────────
• Optimized for timestamp-ordered data
• Compression for sequential values
• Automatic data retention/rollup
• Good for: Metrics, IoT, logs
• Writes: ~100,000+/sec

KEY-VALUE (Redis, DynamoDB)
─────────────────────────────────────────────────────────────
• Simple put/get operations
• In-memory or SSD-optimized
• Horizontal scaling built-in
• Good for: Counters, sessions, caches
• Writes: ~100,000+/sec
""")

## 🔧 Index Management for Writes

In [ ]:
def benchmark_inserts(table_name: str, num_rows: int) -> float:
    conn = get_connection()
    cursor = conn.cursor()
    
    start = time.time()
    for i in range(num_rows):
        cursor.execute(
            f"INSERT INTO {table_name} (event_type, user_id, payload) VALUES (%s, %s, %s)",
            ('benchmark', i % 1000, '{"test": true}')
        )
    conn.commit()
    elapsed = time.time() - start
    
    conn.close()
    return elapsed

conn = get_connection()
cursor = conn.cursor()

cursor.execute("""
    CREATE TABLE IF NOT EXISTS events_no_index (
        id SERIAL PRIMARY KEY,
        event_type VARCHAR(50),
        user_id INTEGER,
        payload JSONB,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
""")

cursor.execute("""
    CREATE TABLE IF NOT EXISTS events_with_indexes (
        id SERIAL PRIMARY KEY,
        event_type VARCHAR(50),
        user_id INTEGER,
        payload JSONB,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
""")

cursor.execute("CREATE INDEX IF NOT EXISTS idx_ewi_type ON events_with_indexes(event_type)")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_ewi_user ON events_with_indexes(user_id)")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_ewi_time ON events_with_indexes(created_at)")

cursor.execute("TRUNCATE events_no_index, events_with_indexes")
conn.commit()
conn.close()

print("✅ Test tables created")

In [ ]:
print("🔧 Index Impact on Write Performance")
print("=" * 60)

num_rows = 1000

print(f"\nInserting {num_rows} rows...")

time_no_index = benchmark_inserts("events_no_index", num_rows)
print(f"\n📊 Table WITHOUT extra indexes:")
print(f"   Time: {time_no_index:.2f}s")
print(f"   Rate: {num_rows/time_no_index:.0f} inserts/sec")

time_with_index = benchmark_inserts("events_with_indexes", num_rows)
print(f"\n📊 Table WITH 3 indexes:")
print(f"   Time: {time_with_index:.2f}s")
print(f"   Rate: {num_rows/time_with_index:.0f} inserts/sec")

slowdown = ((time_with_index - time_no_index) / time_no_index) * 100
print(f"\n💡 Indexes added {slowdown:.1f}% overhead to writes!")

## 📦 Bulk Insert Techniques

In [ ]:
print("📦 Bulk Insert Comparison")
print("=" * 60)

conn = get_connection()
cursor = conn.cursor()
cursor.execute("TRUNCATE events_no_index")
conn.commit()
conn.close()

num_rows = 5000
data = [('bulk_test', i % 1000, '{"batch": true}') for i in range(num_rows)]

print(f"\nInserting {num_rows} rows with different methods...")

conn = get_connection()
cursor = conn.cursor()
start = time.time()
for row in data:
    cursor.execute(
        "INSERT INTO events_no_index (event_type, user_id, payload) VALUES (%s, %s, %s)",
        row
    )
conn.commit()
time_individual = time.time() - start
cursor.execute("TRUNCATE events_no_index")
conn.commit()
conn.close()

print(f"\n1️⃣ Individual INSERTs:")
print(f"   Time: {time_individual:.2f}s")
print(f"   Rate: {num_rows/time_individual:.0f} inserts/sec")

In [ ]:
conn = get_connection()
cursor = conn.cursor()
start = time.time()
psycopg2.extras.execute_batch(
    cursor,
    "INSERT INTO events_no_index (event_type, user_id, payload) VALUES (%s, %s, %s)",
    data,
    page_size=100
)
conn.commit()
time_batch = time.time() - start
cursor.execute("TRUNCATE events_no_index")
conn.commit()
conn.close()

print(f"\n2️⃣ execute_batch (page_size=100):")
print(f"   Time: {time_batch:.2f}s")
print(f"   Rate: {num_rows/time_batch:.0f} inserts/sec")
print(f"   Speedup: {time_individual/time_batch:.1f}x faster")

In [ ]:
from io import StringIO

conn = get_connection()
cursor = conn.cursor()

csv_data = StringIO()
for row in data:
    csv_data.write(f"{row[0]}\t{row[1]}\t{row[2]}\n")
csv_data.seek(0)

start = time.time()
cursor.copy_from(
    csv_data,
    'events_no_index',
    columns=('event_type', 'user_id', 'payload')
)
conn.commit()
time_copy = time.time() - start
conn.close()

print(f"\n3️⃣ COPY (bulk load):")
print(f"   Time: {time_copy:.2f}s")
print(f"   Rate: {num_rows/time_copy:.0f} inserts/sec")
print(f"   Speedup: {time_individual/time_copy:.1f}x faster")

print("\n" + "=" * 60)
print("📊 Summary:")
print(f"   Individual:    {num_rows/time_individual:>8.0f} rows/sec")
print(f"   Batch:         {num_rows/time_batch:>8.0f} rows/sec")
print(f"   COPY:          {num_rows/time_copy:>8.0f} rows/sec")
print("\n💡 COPY is 10-100x faster for bulk loads!")

## 🎛️ Write Optimization Strategies

In [ ]:
print("🎛️ PostgreSQL Write Optimization Strategies")
print("=" * 60)
print("""
1. REDUCE INDEX OVERHEAD
─────────────────────────────────────────────────────────────
   • Drop unnecessary indexes
   • Use partial indexes (WHERE clause)
   • Defer index creation for bulk loads
   
   -- Drop during bulk load
   DROP INDEX idx_events_type;
   -- ... bulk insert ...
   CREATE INDEX idx_events_type ON events(event_type);

2. BATCH COMMITS
─────────────────────────────────────────────────────────────
   • Commit every N rows instead of every row
   • Trade durability for speed
   
   -- Instead of commit after each INSERT
   BEGIN;
   INSERT ...; INSERT ...; INSERT ...;  -- 1000 rows
   COMMIT;

3. DISABLE CONSTRAINTS TEMPORARILY
─────────────────────────────────────────────────────────────
   • Foreign key checks have overhead
   • Disable during trusted bulk loads
   
   SET session_replication_role = 'replica';  -- Disable FK
   -- ... bulk insert ...
   SET session_replication_role = 'origin';   -- Re-enable

4. TUNE WAL SETTINGS
─────────────────────────────────────────────────────────────
   • synchronous_commit = off (risky but fast)
   • wal_buffers = larger value
   • checkpoint_completion_target = 0.9
""")

## 🧪 Quick Quiz

1. **Why is Cassandra faster for writes than PostgreSQL?**

2. **When should you drop indexes before bulk loading?**

3. **What's the trade-off with `synchronous_commit = off`?**

In [ ]:
print("📝 Quiz Answers")
print("=" * 50)
print()
print("1. Cassandra's write advantage:")
print("   - Append-only (sequential writes)")
print("   - No in-place updates (no seeks)")
print("   - Compaction happens in background")
print("   - Trade-off: Slower reads, eventual consistency")
print()
print("2. When to drop indexes:")
print("   - Large bulk loads (>100k rows)")
print("   - When you control all the data")
print("   - Recreate index after load (faster!)")
print()
print("3. synchronous_commit = off:")
print("   - Writes return before WAL flush")
print("   - Risk: Lose last ~200ms of data on crash")
print("   - Use for: Analytics, logs (not transactions!)")

## 📚 Summary

### Key Takeaways

1. **Choose the right database** - Log-structured for write-heavy
2. **Indexes hurt writes** - Only index what you query
3. **Bulk > Individual** - COPY is 10-100x faster
4. **Batch commits** - Reduce transaction overhead
5. **Know your trade-offs** - Speed vs durability

### Next Up

In **Notebook 3**, we'll learn sharding and partitioning:
- Horizontal sharding strategies
- Choosing partition keys
- Avoiding hot spots